Task 1 - Investigación
Además de las cámaras de seguridad con YOLO, los organizadores del Akihabara Fest quieren instalar
"Cabinas Fotográficas de Realidad Aumentada (AR)" para los asistentes VIP. El objetivo de estas cabinas es
doble:
1. Eliminar automáticamente el fondo real de la convención y reemplazarlo por escenarios de anime
(ej. La Aldea Oculta de la Hoja de Naruto o Neo-Tokyo de Akira), simulando un "Chroma Key" (pantalla verde) perfecto sin usar tela verde.
2. Identificar específicamente la espada o báculo mágico que sostiene el usuario para superponerle un efecto de "brillo de energía" digital, diferenciándolo de las armas de otros amigos que estén en la misma foto.
# ______________________________________________________
La mesa directiva no sabe qué modelo de IA usar y le pide a usted investigar dos arquitecturas famosas: U-Net y Mask R-CNN.
Entonces, realice una investigación en artículos o documentación técnica oficial y redacte un reporte ejecutivo (máximo 2 página) respondiendo con rigor técnico a lo siguiente:
1. Diferencia Fundamental: Defina con sus propias palabras la diferencia exacta a nivel de píxel entre
la Segmentación Semántica y la Segmentación de Instancia.
2. El Caso U-Net: Explique por qué la arquitectura U-Net (Segmentación Semántica) es ideal y altamente
eficiente para la tarea 1 (separar a todos los humanos del fondo), pero explique técnicamente por
qué fracasaría si intentamos usarla para la tarea 2 si hay dos réplicas de espadas idénticas cruzadas
en la foto.
3. El Caso Mask R-CNN: Explique cómo la arquitectura Mask R-CNN (Segmentación de Instancia)
resuelve el problema anterior basándose en su naturaleza de "dos etapas" (recordando que hereda
de Faster R-CNN). ¿Por qué Mask R-CNN sí podría iluminar una espada de rojo y otra de azul, incluso
si se están tocando en la imagen?



# **Evaluación de Arquitecturas para Cabinas AR**

# ---

## **1. Diferencia Fundamental entre Segmentación Semántica y Segmentación de Instancia**

La segmentación de imágenes consiste en asignar etiquetas a cada píxel de una imagen. Existen dos enfoques principales: **segmentación semántica** y **segmentación de instancia**, cuya diferencia radica en el nivel de discriminación entre objetos.

La **segmentación semántica** asigna una clase a cada píxel de la imagen, sin distinguir entre objetos individuales. Es decir, todos los píxeles pertenecientes a una misma categoría como el de una persona, reciben la misma etiqueta, pese de cuántas personas haya en la imagen. Desde un punto de vista formal, esto puede representarse como una función que asigna a cada píxel una clase:

f(x, y) → c

donde (x, y) son las coordenadas del píxel y c es la categoría asignada.

En contraste, la **segmentación de instancia** también clasifica cada píxel pero también identifica a qué objeto específico pertenece. Esto implica que dos objetos de la misma clase se consideran entidades separadas. Formalmente:

f(x, y) → (c, i)

donde i representa la instancia individual.

Para poder identificar podemos analizarlo en forma de pregunta:

* Segmentación semántica responde: **¿qué es este píxel?**
* segmentación de instancia responde a: **¿qué es este píxel y a qué objeto específico pertenece?**

---

## **2. Evaluación de U-Net para la Separación de Humanos del Fondo**

La arquitectura **U-Net** es un modelo de segmentación semántica ampliamente utilizado, especialmente en tareas que requieren alta precisión a nivel de píxel. Su estructura tipo encoder-decoder con conexiones de salto permite capturar tanto contexto global como detalles más puntuales.

### **2.1 Ventajas Técnicas de U-Net para la Eliminación de Fondo**

U-Net resulta preferible para la tarea de separar personas del fondo en las cabinas AR porque:

* Clasifica todos los píxeles en una sola pasada, lo que permite distinguir claramente entre “persona” y “fondo”.
* Las *skip connections* transfieren información de alta resolución desde el encoder al decoder, mejorando la precisión en los bordes.
* *Tiene una alta calidad para efectos de realidad aumentada, donde errores en los bordes generan artefactos visuales.

 U-Net entonces puede generar máscaras limpias y precisas de los usuarios, permitiendo reemplazar el fondo real por escenarios virtuales sin necesidad de chroma key físico.

---

### **2.2 Limitaciones de U-Net en la Identificación de Objetos Individuales (Tarea 2)**


Por ejemplo, si dos usuarios sostienen espadas iguales:

* U-Net clasificará todos los píxeles como “espada”.
* Si las espadas están cerca o se cruzan, se generará como una única región segmentada.

Esto es porque el modelo no tiene mecanismos para separar objetos individuales, sino únicamente clases. Entonces:

* No es posible diferenciar a qué espada pertenece cada píxel.
* No se pueden aplicar efectos distintos.
* Se pierde la correspondencia entre objeto y usuario.

Entonces no es ideal para identificación individual.

---

## **3. Evaluación de Mask R-CNN para Segmentación de Instancias (Tarea 2)**

La arquitectura **Mask R-CNN** extiende el modelo Faster R-CNN incorporando una rama adicional para segmentación de instancias. Su diseño en dos etapas le permite identificar objetos individuales y generar máscaras precisas para cada uno.

---

### **3.1 Naturaleza de Dos Etapas en Mask R-CNN**

Mask R-CNN opera asi:

**Etapa 1:**
El modelo genera regiones candidatas (bounding boxes) donde es probable que existan objetos. Cada región corresponde potencialmente a un objeto individual.

**Etapa 2:**
Para cada región propuesta:

* Se clasifica el objeto (persona, espada, etc.).
* Se ajusta la bounding box.
* Se genera una máscara binaria específica para esa instancia.

Asi logra separar los objetos antes de realizar la segmentación fina.

---

### **3.2 Resolución del Problema de Espadas Superpuestas**

Mask R-CNN puede:

* Detectar múltiples objetos de la misma clase como instancias separadas.
* Generar máscaras independientes para cada objeto.
* Mantener la identidad individual incluso en casos de contacto o superposición.

En el caso de dos espadas cruzadas:

* El modelo asigna una región distinta a cada espada.
* Genera dos máscaras separadas.
* Permite aplicar efectos visuales diferenciados a cada una.

Esto ocurre  porque la segmentación se realiza **después de la detección de instancias**, lo que logra la individualidad de los objetos.

---

## **4. Conclusión y Recomendación Técnica**

Entonces se concluye que:

* **U-Net** es altamente eficiente para tareas de segmentación global, como la eliminación de fondo y la detección de personas.
* **Mask R-CNN** es indispensable para tareas que requieren distinguir objetos individuales, especialmente en contextos con múltiples instancias similares.

### **Recomendación**

Se recomienda implementar un **pipeline híbrido**:

1. U-Net para segmentación rápida de personas y eliminación de fondo.
2. Mask R-CNN para detección y segmentación de instancias de objetos (espadas, báculos).






## **Referencias**

* Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation*.
  Disponible en: [https://arxiv.org/abs/1505.04597](https://arxiv.org/abs/1505.04597)

* He, K., Gkioxari, G., Dollár, P., & Girshick, R. (2017). *Mask R-CNN*.
  Disponible en: [https://arxiv.org/abs/1703.06870](https://arxiv.org/abs/1703.06870)

* Girshick, R. (2015). *Fast R-CNN*.
  Disponible en: [https://arxiv.org/abs/1504.08083](https://arxiv.org/abs/1504.08083)

* Ren, S., He, K., Girshick, R., & Sun, J. (2015). *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*.
  Disponible en: [https://arxiv.org/abs/1506.01497](https://arxiv.org/abs/1506.01497)
